# SME-Financial-Decision--Risk-Prediction-Dataset

In [ ]:
A project to evaluate loan performance across SME sectors and assess credit risk metrics

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
## Load the data as a data frame by using URL
fin_data_url="Finanical_data_sme.csv"
df = pd.read_csv(fin_data_url)
print(f"Dataset Loaded. Shape: {df.shape}")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
# descriptive statistics
df.describe()

In [ ]:
df.info()

In [ ]:
# Columns with missing values 
# Literacy_Credit_Knowledge       9500 non-null 
# Assessment_Expert_Consultation  9500 non-null
# Decision_Strategic_Alignment    9500 non-null
# Analysis_Benchmarking           9502 non-null
# FFinancial_Distress              9500 non-null 
null_cols = ['Literacy_Credit_Knowledge', 'Assessment_Expert_Consultation', 
             'Decision_Strategic_Alignment', 'Analysis_Benchmarking', 'Financial_Distress']

In [ ]:
# Fill missing values with median values
for col in null_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
# Count uique values
df["Industry_Sector"].nunique()

In [ ]:
# Get unique values in "Industry_Sector"
df["Industry_Sector"].unique()

In [ ]:
# Count Frequency of Each Unique Value
# Excludes NaN
df["Industry_Sector"].value_counts()

In [ ]:
# Get unique values in "SME_Type"
df["SME_Type"].nunique()


In [ ]:
# Map Industry_Sector to mumeric values to textual lables
sector_map = {1: 'Technology', 2: 'Retail', 3: 'Manufacturing', 4: 'Agriculture', 5: 'Services'}
df['Industry_Sector_Label'] = df['Industry_Sector'].map(sector_map)

#### The composite score translates raw categorical values into a single, quantifiable **risk profile** for each SME.

### Formula Breakdown

$$\text{Composite Risk Score} = (2 \times \text{Financial Distress}) + \text{Risk Taking Willingness} - \text{Risk Mitigation Strategies}$$

In [ ]:
# Risk Score: Higher Financial Distress & Risk Taking Willingness, Lower Risk Mitigation
df['Composite_Risk_Score'] = df['Financial_Distress'] * 2 + df['Risk_Taking_Willingness'] - df['Risk_Mitigation_Strategies']

# Profitability Proxy: High Revenue + High Liquidity + Strategic Alignment
rev_map = {'Low': 1, 'Medium': 2, 'High': 3}
df['Revenue_Numeric'] = df['Annual_Revenue_Category'].map(rev_map)
df['Profitability_Index'] = df['Revenue_Numeric'] * df['Liquidity_Stability']

# Capital Proxy: Decision_Capital_Allocation scale combined with SME_Size
df['Capital_Scale_Index'] = df['Decision_Capital_Allocation'] * df['SME_Size_Category']

print("Data Cleaning and Feature Engineering Complete.")

In [ ]:
# ==============================================================================
# Highest / Lowest Risk Sector
# ==============================================================================
plt.figure(figsize=(10, 6))
risk_by_sector = df.groupby('Industry_Sector_Label')['Composite_Risk_Score'].mean().sort_values()
sns.barplot(x=risk_by_sector.values, y=risk_by_sector.index, palette='Reds_r')
plt.title('Average Risk Profile by Sector (Lower = Safer)', fontsize=12, fontweight='bold')
plt.xlabel('Composite Risk Index')
plt.tight_layout()
plt.savefig('images/risk_by_sector.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# ==============================================================================
# Highest / Lowest Capital Allocation
# ==============================================================================
plt.figure(figsize=(10, 6))
capital_by_sector = df.groupby('Industry_Sector_Label')['Capital_Scale_Index'].mean().sort_values()
sns.barplot(x=capital_by_sector.values, y=capital_by_sector.index, palette='Blues_r')
plt.title('Average Capital Index by Sector', fontsize=12, fontweight='bold')
plt.xlabel('Capital Index')
plt.tight_layout()
plt.savefig('images/capital_by_sector.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# ==============================================================================
# Most Profitability Sectors
# ==============================================================================
plt.figure(figsize=(10, 6))
profit_by_sector = df.groupby('Industry_Sector_Label')['Profitability_Index'].mean().sort_values(ascending=False)
sns.barplot(x=profit_by_sector.values, y=profit_by_sector.index, palette='Greens_r')
plt.title('Profitability Index by Sector', fontsize=12, fontweight='bold')
plt.xlabel('Profitability Index')
plt.tight_layout()
plt.savefig('images/profitability_by_sector.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# ==============================================================================
# Loan Approval vs Financial Distress Across Sectors
# ==============================================================================
plt.figure(figsize=(10, 6))
approval_distress = df.groupby('Industry_Sector_Label')[['Decision_Loan_Approval', 'Financial_Distress']].mean()
ax = approval_distress.plot(kind='bar', figsize=(10, 6), rot=45, color=['#2ca02c', '#d62728'])
plt.title('Loan Approval Rate vs Financial Distress Rate', fontsize=12, fontweight='bold')
plt.ylabel('Rate (0.0 to 1.0)')
plt.tight_layout()
plt.savefig('images/approval_vs_distress_by_sector.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# ==============================================================================
# Correlation Matrix
# ==============================================================================
plt.figure(figsize=(10, 6))

# Select key numerical performance indicators
corr_cols = [
    'SME_Age', 'Literacy_Accounting', 'Literacy_Credit_Knowledge', 
    'Risk_Mitigation_Strategies', 'Decision_Loan_Approval', 
    'Liquidity_Stability', 'Financial_Distress', 'Profitability_Index', 'Capital_Scale_Index'
]

corr_matrix = df[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, linewidths=0.5)
plt.title("Correlation Matrix of Key SME Performance Drivers", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_matrix2.png', dpi=300)
plt.show()